# Diabetes Risk Predictor
**Predictive Analytics Mini Project**

This notebook demonstrates an end-to-end binary classification workflow using the Pima Indians Diabetes dataset (768 records, 8 clinical predictors). It compares Logistic Regression, Linear Discriminant Analysis (LDA), and Quadratic Discriminant Analysis (QDA).

> **Educational-use disclaimer:** This model is not a medical diagnostic tool. It must not be used to make healthcare decisions.

## 1. Import libraries and set up

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay)

RANDOM_STATE = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
(DATA_DIR := ROOT / "data").mkdir(exist_ok=True)
(IMG_DIR := ROOT / "images").mkdir(exist_ok=True)
(RESULT_DIR := ROOT / "results").mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")

## 2. Load the dataset

In [ ]:
# The original Pima dataset is publicly mirrored here. The file has no header row.
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
local_csv = DATA_DIR / "diabetes.csv"

if local_csv.exists():
    df = pd.read_csv(local_csv)
else:
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    df = pd.read_csv(url, header=None, names=columns)
    df.to_csv(local_csv, index=False)

# If a headered CSV was supplied, retain its existing column names.
if "Outcome" not in df.columns and df.shape[1] == 9:
    df.columns = columns

print("Shape:", df.shape)
display(df.head())
display(df.describe().T)

## 3. Data quality checks and preprocessing

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum())

# In this dataset, zero is not physiologically plausible for these measurements;
# treat it as missing. Zero pregnancies and Outcome=0 remain valid.
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

print("Missing values after identifying invalid zeros:")
display(df.isna().sum())

X = df.drop(columns="Outcome")
y = df["Outcome"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="Outcome")
plt.title("Diabetes outcome distribution")
plt.xlabel("Outcome (0 = negative, 1 = positive)")
plt.tight_layout()
plt.savefig(IMG_DIR / "outcome_distribution.png", dpi=150)
plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.savefig(IMG_DIR / "correlation_heatmap.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="Outcome", y="Glucose", ax=axes[0])
sns.boxplot(data=df, x="Outcome", y="BMI", ax=axes[1])
axes[0].set_title("Glucose by outcome")
axes[1].set_title("BMI by outcome")
plt.tight_layout()
plt.savefig(IMG_DIR / "feature_boxplots.png", dpi=150)
plt.show()

## 5. Train and compare Logistic Regression, LDA and QDA

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "LDA": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearDiscriminantAnalysis())
    ]),
    "QDA": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", QuadraticDiscriminantAnalysis(reg_param=0.1))
    ])
}

scoring = {"accuracy": "accuracy", "precision": "precision",
           "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    cv_rows.append({
        "Model": name,
        **{metric: scores[f"test_{metric}"].mean() for metric in scoring}
    })

cv_results = pd.DataFrame(cv_rows).sort_values("roc_auc", ascending=False)
display(cv_results.round(3))
cv_results.to_csv(RESULT_DIR / "cross_validation_comparison.csv", index=False)

ax = cv_results.set_index("Model")[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0
)
ax.set_title("5-fold cross-validation comparison")
ax.set_ylabel("Mean score")
plt.tight_layout()
plt.savefig(IMG_DIR / "model_comparison.png", dpi=150)
plt.show()

## 6. Hyperparameter tuning

In [ ]:
# Tune Logistic Regression as a transparent, interpretable baseline.
# The search is performed only on training data using stratified CV.
lr_pipeline = models["Logistic Regression"]

grid = GridSearchCV(
    lr_pipeline,
    param_grid={"model__C": [0.01, 0.1, 1, 10, 100],
                "model__class_weight": [None, "balanced"]},
    scoring="roc_auc", cv=cv, n_jobs=-1
)
grid.fit(X_train, y_train)
print("Grid-search best parameters:", grid.best_params_)
print("Grid-search best CV ROC-AUC:", round(grid.best_score_, 3))

random_search = RandomizedSearchCV(
    lr_pipeline,
    param_distributions={"model__C": np.logspace(-2, 2, 30),
                         "model__class_weight": [None, "balanced"]},
    n_iter=12, scoring="roc_auc", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1
)
random_search.fit(X_train, y_train)
print("Random-search best parameters:", random_search.best_params_)
print("Random-search best CV ROC-AUC:", round(random_search.best_score_, 3))

# Select the better CV search result; test set remains untouched until next section.
best_model = grid.best_estimator_ if grid.best_score_ >= random_search.best_score_ else random_search.best_estimator_

## 7. Final evaluation on the unseen test set

In [ ]:
# This is the one final evaluation on the held-out test set.
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
display(pd.DataFrame([test_metrics]).round(3))
pd.DataFrame([test_metrics]).to_csv(RESULT_DIR / "test_metrics.csv", index=False)

print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Confusion matrix – held-out test set")
plt.tight_layout()
plt.savefig(IMG_DIR / "confusion_matrix.png", dpi=150)
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC curve – held-out test set")
plt.tight_layout()
plt.savefig(IMG_DIR / "roc_curve.png", dpi=150)
plt.show()

## 8. Odds ratios for Logistic Regression

In [ ]:
# Fit the tuned Logistic Regression model coefficients on the training data.
# Coefficients correspond to standardized features because StandardScaler is used.
lr_final = best_model
if "model" in lr_final.named_steps and isinstance(lr_final.named_steps["model"], LogisticRegression):
    coefficients = lr_final.named_steps["model"].coef_[0]
    odds_ratios = np.exp(coefficients)
    odds_table = pd.DataFrame({
        "Feature": X.columns,
        "Standardized coefficient": coefficients,
        "Odds ratio (per 1 SD increase)": odds_ratios
    }).sort_values("Odds ratio (per 1 SD increase)", ascending=False)
    display(odds_table.round(3))
    odds_table.to_csv(RESULT_DIR / "odds_ratios.csv", index=False)
else:
    print("Selected search result was not Logistic Regression; odds ratios are shown from a separately fitted Logistic Regression model.")
    odds_model = models["Logistic Regression"].fit(X_train, y_train)
    coefficients = odds_model.named_steps["model"].coef_[0]
    odds_table = pd.DataFrame({
        "Feature": X.columns,
        "Standardized coefficient": coefficients,
        "Odds ratio (per 1 SD increase)": np.exp(coefficients)
    }).sort_values("Odds ratio (per 1 SD increase)", ascending=False)
    display(odds_table.round(3))
    odds_table.to_csv(RESULT_DIR / "odds_ratios.csv", index=False)

## 9. Conclusion

This project compares three classical classification algorithms for diabetes-risk prediction. Cross-validation is used for model comparison and hyperparameter selection, while a held-out test set is reserved for final evaluation.

Interpret the final metrics together: recall is important for understanding missed positive cases, while precision indicates how many predicted positive cases are actually positive. Odds ratios describe associations learned by Logistic Regression; they do not establish causation.

**Limitations:** The dataset is relatively small, historical, and specific to a particular population. Model performance may not generalize to other populations or clinical settings. This project is for learning only and is not a medical diagnostic system.